<a href="https://colab.research.google.com/github/Pranayshukla0610/basic-python/blob/main/Amazon_Sales_Report_using_Numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
import os

In [ ]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"pranay061020","key":"448c8cce58030214e5b75e66581529c2"}'}

In [ ]:
os.environ['KAGGLE_USERNAME'] = "pranay061020"
os.environ['KAGGLE_KEY'] = "448c8cce58030214e5b75e66581529c2"

In [ ]:
os.system("kaggle datasets download -d thedevastator/unlock-profits-with-e-commerce-sales-data")
os.system("unzip unlock-profits-with-e-commerce-sales-data.zip")

0

In [ ]:
df = pd.read_csv("/content/Amazon Sale Report.csv")

/tmp/ipython-input-2946482716.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/Amazon Sale Report.csv")


In [ ]:
df.head()

,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,...,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,...,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,...,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,...,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,...,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,...,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  object 
 2   Date                128975 non-null  object 
 3   Status              128975 non-null  object 
 4   Fulfilment          128975 non-null  object 
 5   Sales Channel       128975 non-null  object 
 6   ship-service-level  128975 non-null  object 
 7   Style               128975 non-null  object 
 8   SKU                 128975 non-null  object 
 9   Category            128975 non-null  object 
 10  Size                128975 non-null  object 
 11  ASIN                128975 non-null  object 
 12  Courier Status      122103 non-null  object 
 13  Qty                 128975 non-null  int64  
 14  currency            121180 non-null  object 
 15  Amount              121180 non-nul

In [ ]:
df.isnull().sum()

,0
index,0
Order ID,0
Date,0
Status,0
Fulfilment,0
Sales Channel,0
ship-service-level,0
Style,0
SKU,0
Category,0


In [ ]:
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df['ship-postal-code'] = pd.to_numeric(df['ship-postal-code'], errors='coerce')

numerical_cols = df[['Qty','Amount','ship-postal-code']]
df = numerical_cols.values

In [ ]:
type(df)

numpy.ndarray

In [ ]:
col_means = np.nanmean(df,axis=0)
inds = np.where(np.isnan(df))
df[inds] = np.take(col_means,inds[1])

In [ ]:
mean = np.mean(df,axis=0)
std = np.std(df,axis=0)
median = np.median(df,axis=0)
variance = np.var(df,axis=0)
min_vals = np.min(df,axis=0)
max_val = np.max(df,axis=0)

In [30]:
Q1 = np.percentile(df[:,1],25)
Q3 = np.percentile(df[:,1],75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

mask = (df[:,1] >= lower) & (df[:,1] <= upper)
clean_df = df[mask]

In [31]:
revenue_per_unit = clean_df[:,1]/(clean_df[:,0]+1)

In [32]:
log_revenue = np.log(clean_df[:,1]+1)

In [33]:
revenue_sq = clean_df[:,1]**2

In [34]:
final_df = np.column_stack((clean_df,revenue_per_unit,log_revenue,revenue_sq))

In [35]:
final_df

array([[0.00000000e+00, 6.47620000e+02, 4.00081000e+05, 6.47620000e+02,
        6.47484703e+00, 4.19411664e+05],
       [1.00000000e+00, 4.06000000e+02, 5.60085000e+05, 2.03000000e+02,
        6.00881319e+00, 1.64836000e+05],
       [1.00000000e+00, 3.29000000e+02, 4.10210000e+05, 1.64500000e+02,
        5.79909265e+00, 1.08241000e+05],
       ...,
       [1.00000000e+00, 6.90000000e+02, 5.00049000e+05, 3.45000000e+02,
        6.53813982e+00, 4.76100000e+05],
       [1.00000000e+00, 1.19900000e+03, 3.89350000e+05, 5.99500000e+02,
        7.09007684e+00, 1.43760100e+06],
       [1.00000000e+00, 6.96000000e+02, 4.92014000e+05, 3.48000000e+02,
        6.54678541e+00, 4.84416000e+05]])

In [36]:
cov_matrix = np.cov(final_df.T)
corr_matrix = np.corrcoef(final_df.T)

In [37]:
cov_matrix

array([[ 9.71463561e-02, -3.98371723e-01, -6.09674122e+02,
        -2.95585898e+01, -1.41877489e-02,  3.26004804e+03],
       [-3.98371723e-01,  5.54290987e+04, -1.70813885e+06,
         2.89012243e+04,  1.49093637e+02,  7.30509803e+07],
       [-6.09674122e+02, -1.70813885e+06,  3.64711338e+10,
        -6.97203530e+05,  1.28309973e+02, -2.59290719e+09],
       [-2.95585898e+01,  2.89012243e+04, -6.97203530e+05,
         2.47294909e+04,  8.08461439e+01,  3.68785196e+07],
       [-1.41877489e-02,  1.49093637e+02,  1.28309973e+02,
         8.08461439e+01,  8.73093824e-01,  1.54220982e+05],
       [ 3.26004804e+03,  7.30509803e+07, -2.59290719e+09,
         3.68785196e+07,  1.54220982e+05,  1.02851677e+11]])

In [38]:
corr_matrix

array([[ 1.00000000e+00, -5.42883120e-03, -1.02425920e-02,
        -6.03063252e-01, -4.87157930e-02,  3.26140742e-02],
       [-5.42883120e-03,  1.00000000e+00, -3.79909042e-02,
         7.80620126e-01,  6.77734731e-01,  9.67501066e-01],
       [-1.02425920e-02, -3.79909042e-02,  1.00000000e+00,
        -2.32154634e-02,  7.19043763e-04, -4.23356910e-02],
       [-6.03063252e-01,  7.80620126e-01, -2.32154634e-02,
         1.00000000e+00,  5.50200859e-01,  7.31240444e-01],
       [-4.87157930e-02,  6.77734731e-01,  7.19043763e-04,
         5.50200859e-01,  1.00000000e+00,  5.14644478e-01],
       [ 3.26140742e-02,  9.67501066e-01, -4.23356910e-02,
         7.31240444e-01,  5.14644478e-01,  1.00000000e+00]])